# SurfaceTarget-CRC — phase 01 — patch 1 7

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.7
#
# Additional MSI Annotation Discovery
# and Research Checkpoint
#
# Objectives:
# 1. Examine related TCGA colorectal studies
# 2. Identify MSI-related clinical attributes
# 3. Identify potential categorical annotations
# 4. Save discovery metadata
# 5. Create a research checkpoint
#
# No additional patient classifications are performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import requests
import json
import re

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CBIO_API = "https://www.cbioportal.org/api"

# Candidate TCGA colorectal studies.
# Some may be unavailable or retired.
# Unavailable studies will be recorded and skipped.

STUDY_IDS = [
    "coadread_tcga",
    "coadread_tcga_pub",
    "coadread_tcga_pan_can_atlas_2018",
    "coad_tcga",
    "read_tcga"
]

# Relevant clinical annotation keywords

KEYWORDS = [
    "MSI",
    "MICROSAT",
    "MANTIS",
    "MISMATCH",
    "MMR"
]

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.7 — ADDITIONAL MSI ANNOTATION DISCOVERY")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load preliminary cohort
# ------------------------------------------------------------

COHORT_PATH = (
    METADATA_DIR /
    "tcga_msi_preliminary_patient_classification.csv"
)

cohort = pd.read_csv(COHORT_PATH)

print("\nPreliminary cohort loaded.")

print("Total primary tumor patients:", len(cohort))

print("\nCurrent classification:")

print(
    cohort["msi_group"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 3. Initialize API session
# ------------------------------------------------------------

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SurfaceTarget-CRC-Research/1.0"
})

# ------------------------------------------------------------
# 4. Investigate available studies
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CBIOPORTAL STUDY DISCOVERY")
print("-" * 65)

all_attributes = []

study_status = {}

for study_id in STUDY_IDS:

    print(f"\nInvestigating: {study_id}")

    endpoint = (
        f"{CBIO_API}/studies/"
        f"{study_id}/clinical-attributes"
    )

    try:

        response = session.get(
            endpoint,
            params={"pageSize": 10000},
            timeout=60
        )

        if response.status_code == 404:

            print("Status: Study unavailable")

            study_status[study_id] = "unavailable"

            continue

        response.raise_for_status()

        attributes = response.json()

        if not isinstance(attributes, list):

            raise ValueError(
                "Unexpected clinical attribute response."
            )

        study_status[study_id] = "available"

        print(
            "Total clinical attributes:",
            len(attributes)
        )

        # --------------------------------------------
        # Identify MSI-related attributes
        # --------------------------------------------

        matching_attributes = []

        for attribute in attributes:

            attribute_id = (
                attribute.get("clinicalAttributeId")
                or attribute.get("id")
                or ""
            )

            display_name = (
                attribute.get("displayName")
                or attribute.get("name")
                or ""
            )

            description = (
                attribute.get("description")
                or ""
            )

            searchable_text = " ".join([
                str(attribute_id),
                str(display_name),
                str(description)
            ]).upper()

            if any(
                keyword in searchable_text
                for keyword in KEYWORDS
            ):

                record = {

                    "study_id": study_id,

                    "attribute_id": attribute_id,

                    "display_name": display_name,

                    "description": description,

                    "data_type": attribute.get(
                        "dataType", "Unknown"
                    ),

                    "patient_attribute": attribute.get(
                        "patientAttribute"
                    )
                }

                matching_attributes.append(record)

                all_attributes.append(record)

        print(
            "MSI-related attributes:",
            len(matching_attributes)
        )

        for attribute in matching_attributes:

            print(
                f"  - {attribute['attribute_id']}"
            )

            print(
                f"    Name: {attribute['display_name']}"
            )

            print(
                f"    Data type: {attribute['data_type']}"
            )

            level = (
                "Patient"
                if attribute["patient_attribute"]
                else "Sample"
            )

            print(f"    Level: {level}")

    except Exception as e:

        print(f"Status: ERROR — {e}")

        study_status[study_id] = (
            f"error: {str(e)}"
        )

# ------------------------------------------------------------
# 5. Create discovery dataframe
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSI ANNOTATION DISCOVERY SUMMARY")
print("-" * 65)

discovery_df = pd.DataFrame(
    all_attributes,
    columns=[
        "study_id",
        "attribute_id",
        "display_name",
        "description",
        "data_type",
        "patient_attribute"
    ]
)

print(
    "\nTotal MSI-related attribute definitions:",
    len(discovery_df)
)

if not discovery_df.empty:

    print("\nAttributes by study:")

    print(
        discovery_df.groupby("study_id")
        .size()
        .to_string()
    )

# ------------------------------------------------------------
# 6. Identify possible categorical MSI fields
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("POTENTIAL CATEGORICAL MSI ANNOTATIONS")
print("-" * 65)

if not discovery_df.empty:

    # These are only candidate fields.
    # Their actual contents and provenance
    # must be inspected before use.

    categorical_candidates = discovery_df[
        discovery_df["data_type"]
        .astype(str)
        .str.upper()
        .eq("STRING")
    ].copy()

    print(
        "\nPotential categorical fields:",
        len(categorical_candidates)
    )

    if not categorical_candidates.empty:

        print(
            categorical_candidates[
                [
                    "study_id",
                    "attribute_id",
                    "display_name"
                ]
            ].to_string(index=False)
        )

    else:

        print(
            "No STRING-type MSI-related fields identified."
        )

else:

    categorical_candidates = discovery_df.copy()

    print(
        "No MSI-related attributes identified."
    )

# ------------------------------------------------------------
# 7. Save discovery results
# ------------------------------------------------------------

discovery_path = (
    METADATA_DIR /
    "tcga_additional_msi_attribute_discovery.csv"
)

discovery_df.to_csv(
    discovery_path,
    index=False
)

categorical_path = (
    METADATA_DIR /
    "tcga_potential_categorical_msi_attributes.csv"
)

categorical_candidates.to_csv(
    categorical_path,
    index=False
)

# ------------------------------------------------------------
# 8. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "retrieved_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "source": "cBioPortal",

    "api": CBIO_API,

    "studies_investigated": STUDY_IDS,

    "study_status": study_status,

    "total_matching_attribute_definitions": int(
        len(discovery_df)
    ),

    "potential_categorical_fields": int(
        len(categorical_candidates)
    ),

    "msi_status_assigned": False,

    "note": (
        "Discovery of additional MSI clinical attributes. "
        "No new classifications assigned. "
        "Annotation contents and provenance require "
        "evaluation before inclusion in the study."
    )
}

provenance_path = (
    METADATA_DIR /
    "tcga_additional_msi_discovery_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 9. Create research checkpoint
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)

timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")

checkpoint_path = (
    CHECKPOINT_DIR /
    f"SurfaceTarget_CRC_Phase1_Checkpoint_{timestamp}.zip"
)

# Include metadata and small research outputs.
# Exclude raw molecular datasets.

backup_directories = [
    "data/metadata",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]

with ZipFile(
    checkpoint_path,
    "w",
    compression=ZIP_DEFLATED
) as zip_file:

    for directory in backup_directories:

        directory_path = (
            PROJECT_ROOT / directory
        )

        if directory_path.exists():

            for file_path in directory_path.rglob("*"):

                if file_path.is_file():

                    archive_name = (
                        file_path.relative_to(PROJECT_ROOT)
                    )

                    zip_file.write(
                        file_path,
                        arcname=str(archive_name)
                    )

checkpoint_mb = (
    checkpoint_path.stat().st_size
    / (1024 ** 2)
)

print("\nCheckpoint created successfully.")

print(
    f"Checkpoint size: {checkpoint_mb:.2f} MB"
)

print("\nCheckpoint location:")
print(checkpoint_path)

# ------------------------------------------------------------
# 10. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.7 SUMMARY")
print("=" * 65)

print("\nFiles saved:")

print(
    "1. tcga_additional_msi_attribute_discovery.csv"
)

print(
    "2. tcga_potential_categorical_msi_attributes.csv"
)

print(
    "3. tcga_additional_msi_discovery_provenance.json"
)

print("\nResearch checkpoint created.")

print("\nNo new MSS labels assigned.")
print("No molecular expression datasets downloaded.")

print("\nPATCH 1.7 COMPLETE")